# Phi-4 Multimodal — DIMER multi-capability tutorial

**Profile:** `MULTI-CAPABILITY`  
**Notebook specification:** DIMER Notebook Specification 1.0  
**Capability:** text-, image-, and audio-conditioned text generation using one pinned Phi-4 Multimodal checkpoint

This notebook is the executable reference path for the repository capability. It exercises the repository's public pipeline API rather than reimplementing model inference. The default sample is demonstration evidence, not a production-quality or benchmark claim.

**Learning objectives:** bootstrap the repository in a fresh runtime, resolve the immutable upstream model revision, acknowledge the custom-code trust boundary, run text/image/audio capabilities through one public API, exercise optional BYOD image input, and export machine-readable outputs plus provenance.


## Prerequisites

Run in a fresh CUDA runtime with enough memory for the roughly 13 GB model snapshot. This reference path uses `eager` attention for portability rather than silently requiring FlashAttention 2; optimized deployments may explicitly opt into `flash_attention_2` after installing a compatible build. Optional BYOD upload is gated off by default. Do not upload confidential or restricted media to a hosted notebook environment unless authorized. Inputs remain in the notebook runtime and are not sent to a third-party inference API.


## 1. Bootstrap the repository and pinned runtime

When no repository checkout exists, this cell clones the repository. Released notebooks default to `main`; automated candidate validation can set `DIMER_TUTORIAL_REF` to an immutable commit or review branch. The repository is installed as a regular (non-editable) package so it is importable in this same runtime; an editable install would only become importable after a restart. Model-facing dependencies are directly pinned. If installation replaces an already imported core package, the cell fails with a restart instruction rather than continuing with mixed versions.

In [ ]:
import importlib
import importlib.metadata
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = 'https://github.com/kurtvalcorza/phi4-multimodal-pipeline.git'
REPO_NAME = 'phi4-multimodal-pipeline'
REPO_REF = os.environ.get('DIMER_TUTORIAL_REF', 'main')
SKIP_INSTALL = os.environ.get('DIMER_NOTEBOOK_CI_PREINSTALLED') == '1'
ROOT = Path.cwd()
if not (ROOT / 'pyproject.toml').exists():
    checkout = ROOT / REPO_NAME
    if not checkout.exists():
        subprocess.run(['git', 'clone', '--filter=blob:none', '-q', REPO_URL, str(checkout)], check=True)
    if REPO_REF != 'main':
        subprocess.run(['git', '-C', str(checkout), 'fetch', '--depth', '1', 'origin', REPO_REF], check=True)
        subprocess.run(['git', '-C', str(checkout), 'checkout', '--detach', 'FETCH_HEAD'], check=True)
    else:
        subprocess.run(['git', '-C', str(checkout), 'checkout', '-q', 'main'], check=True)
        subprocess.run(['git', '-C', str(checkout), 'pull', '--ff-only', '-q', 'origin', 'main'], check=True)
    os.chdir(checkout)
    ROOT = Path.cwd()

if not SKIP_INSTALL:
    tracked = {'torch': 'torch', 'transformers': 'transformers'}
    # Distribution versions of core packages that are already imported, captured before
    # installation. Metadata is compared with metadata afterwards: torch.__version__ carries a
    # local build label (for example 2.6.0+cu124) that the distribution version omits.
    def _installed_version(distribution):
        try:
            return importlib.metadata.version(distribution)
        except importlib.metadata.PackageNotFoundError:
            return None
    loaded = {distribution: _installed_version(distribution) for distribution, module in tracked.items() if module in sys.modules}
    # Non-editable install: an editable (.pth) install is not importable until the
    # interpreter restarts, which a fresh hosted runtime cannot do mid-notebook.
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', f'{ROOT}[tutorial]'], check=True)
    importlib.invalidate_caches()
    stale = []
    for distribution, before in loaded.items():
        installed = _installed_version(distribution)
        if before is not None and before != installed:
            stale.append(f'{distribution}: loaded={before}, installed={installed}')
    if stale:
        raise RuntimeError('Core dependencies changed while older modules were loaded: ' + '; '.join(stale) + '. Restart the runtime, then rerun from the top.')

REPO_SHA = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
import platform, torch, transformers
print({'repository': str(ROOT), 'repository_revision': REPO_SHA, 'requested_ref': REPO_REF, 'python': platform.python_version(), 'torch': torch.__version__, 'transformers': transformers.__version__, 'cuda': torch.cuda.is_available()})
assert torch.cuda.is_available(), 'A CUDA GPU runtime is required for the release-reference path.'

## 2. Resolve the pinned model and acknowledge the trust boundary

This upstream model requires custom Python code. The repository loader refuses implicit trust; this cell opts in only after printing the immutable code/weight revision. The reference path explicitly selects eager attention so it does not silently depend on a locally compiled FlashAttention package.

In [ ]:
from phi4_multimodal_pipeline import MODEL_ID, MODEL_REVISION, Phi4MultimodalPipeline
print({'model_id': MODEL_ID, 'revision': MODEL_REVISION, 'trust_remote_code': 'explicitly enabled at pinned revision', 'attention': 'eager'})
pipe = Phi4MultimodalPipeline.from_pretrained(allow_remote_code=True, device='cuda', attention_implementation='eager')

## 3. Capability A — text-only generation

Deterministic decoding is used for the tutorial. The result is generated text, not a calibrated confidence statement.

In [ ]:
text_result = pipe.generate('In two sentences, explain what automatic speech recognition does.', max_new_tokens=96, temperature=0.0)
print(text_result['text'])

## 4. Capability B — image + text

The default image is the public stop-sign photograph used in Microsoft's upstream sample. BYOD can replace it by setting `USE_BYOD_IMAGE=True`.

In [ ]:
from io import BytesIO
from urllib.request import urlopen
from PIL import Image
USE_BYOD_IMAGE = False
if USE_BYOD_IMAGE:
    from google.colab import files
    uploaded = files.upload()
    image = Image.open(next(iter(uploaded))).convert('RGB')
else:
    image = Image.open(BytesIO(urlopen('https://www.ilankelman.org/stopsigns/australia.jpg', timeout=30).read())).convert('RGB')
image_result = pipe.generate('Describe the most prominent traffic sign in the image.', images=[image], max_new_tokens=96, temperature=0.0)
print(image_result['text'])

## 5. Capability C — audio + text

The default path uses a public LibriSpeech sample decoded with the pinned `soundfile` dependency. The processor receives the `(audio_array, sampling_rate)` tuple used by the upstream inference contract. The output is generated transcription text; no universal accuracy claim is inferred from this one example.

In [ ]:
import io

import soundfile as sf
from datasets import Audio, load_dataset

# The audio bytes are decoded with the pinned soundfile dependency, matching the upstream
# sample; the datasets audio feature would need torchcodec/FFmpeg, which is not pinned here.
ds = load_dataset('hf-internal-testing/librispeech_asr_dummy', 'clean', split='validation')
ds = ds.cast_column('audio', Audio(decode=False))
waveform, sampling_rate = sf.read(io.BytesIO(ds[0]['audio']['bytes']), dtype='float32')
audio = (waveform, sampling_rate)
audio_result = pipe.generate('Generate a transcription of the attached speech.', audios=[audio], max_new_tokens=128, temperature=0.0)
print(audio_result['text'])

## 6. Export outputs and provenance

Each capability is exported separately while preserving repository revision, shared model identity, trust boundary, attention backend, and decoding settings.

In [ ]:
import json
os.makedirs('outputs', exist_ok=True)
payload = {
    'repository_revision': REPO_SHA,
    'model_id': MODEL_ID,
    'model_revision': MODEL_REVISION,
    'capabilities': {'text': text_result, 'image': image_result, 'audio': audio_result},
    'runtime': {
        'python': platform.python_version(),
        'torch': torch.__version__,
        'transformers': transformers.__version__,
        'device': pipe.device,
        'attention_implementation': pipe.attention_implementation,
    },
}
with open('outputs/phi4_multimodal_results.json', 'w', encoding='utf-8') as handle:
    json.dump(payload, handle, indent=2, ensure_ascii=False)
print('outputs/phi4_multimodal_results.json')

## Interpretation and limits

The three outputs share one generative model but have different evidence sources and failure modes. Non-empty text only establishes that the inference path executed. Image and audio interpretations can be wrong, and no calibrated answer confidence is returned. The tutorial deliberately does not expose fine-tuning or claim that upstream benchmark results were reproduced.

Successful execution proves that the recorded repository revision can acquire the pinned model, execute the explicitly acknowledged pinned custom-code boundary, run the demonstrated public pipeline capabilities, and emit the shown machine-readable outputs in the tested runtime. It does **not** establish benchmark superiority, deployment calibration, safety for high-consequence decisions, or production fitness on an unseen domain.

## References

- Repository README: `../README.md`
- Repository model card: `../MODEL_CARD.md`
- Upstream model: https://huggingface.co/microsoft/Phi-4-multimodal-instruct
- Pinned upstream sample code: https://huggingface.co/microsoft/Phi-4-multimodal-instruct/blob/93f923e1a7727d1c4f446756212d9d3e8fcc5d81/sample_inference_phi4mm.py
- Technical report: https://arxiv.org/abs/2503.01743
